# M4 handoff
Run initialization cells in order. No QA authoring cells are included. Read README_FIRST.md for external data and environment requirements.

## Initialization 1

In [ ]:
from pathlib import Path
import sys, os, json, hashlib, shutil, time, uuid, platform

assert sys.version_info[:2] == (3, 11), "Select Python (cs5481-m4)"
BASE = Path(os.environ.get("M4_WORK_BASE", "D:/CS5481_work")).resolve()
COMMIT = "a0482efea2f523f6a918fa917e4c9cc85d32ec57"
PROJECT_ROOT = Path.cwd().resolve()
DATA_ROOT = Path(os.environ.get("M4_DATA_ROOT", str(BASE / "m3_data"))).resolve()
RUN_DB = Path(os.environ.get("M4_RUN_DB", str(BASE / "runtime_chroma_v030"))).resolve()
EXPECTED_ARCHIVE_SHA = "89776eabda1e3dd4b1db8f096b970c0fb156357b40017dc8f78c49178f3ff586"
os.environ["HF_HOME"] = str(BASE / "model_cache")
BASE.mkdir(parents=True, exist_ok=True)
for path in [PROJECT_ROOT, DATA_ROOT, PROJECT_ROOT / "scripts/m4_retrieval/retriever.py"]:
    if not path.exists():
        raise FileNotFoundError(path)

def sha256_file(path):
    value = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(4 * 1024 * 1024), b""):
            value.update(block)
    return value.hexdigest()

print("Python:", sys.executable)
print("Project:", PROJECT_ROOT)
print("Paths ready")


## Initialization 2

In [ ]:
import chromadb
import numpy as np
from importlib.metadata import version

assert version("chromadb") == "1.5.9", "Install the required version in section 3 and restart the kernel"
db_ready = RUN_DB / ".m4_copy_ready.json"
if not db_ready.exists():
    if RUN_DB.exists():
        raise RuntimeError("Runtime database directory exists without a completion marker; inspect it before proceeding. Do not overwrite it.")
    shutil.copytree(DATA_ROOT / "data/vector_store/chroma", RUN_DB)
    db_ready.write_text(json.dumps({"archive_sha256":EXPECTED_ARCHIVE_SHA}), encoding="utf-8")
assert json.loads(db_ready.read_text())["archive_sha256"] == EXPECTED_ARCHIVE_SHA

def verify_persisted_distance(db_dir, collection_name, metadata, expected):
    """Verify the active vector index in the pinned Chroma 1.5.9 schema."""
    import sqlite3
    from contextlib import closing

    db_path = (Path(db_dir) / "chroma.sqlite3").resolve()
    with closing(sqlite3.connect(db_path.as_uri() + "?mode=ro", uri=True)) as db:
        row = db.execute(
            "SELECT schema_str FROM collections WHERE name = ?",
            (collection_name,),
        ).fetchone()
    if row is None or not row[0]:
        raise RuntimeError(f"Missing persisted vector schema: {collection_name}")
    schema = json.loads(row[0])
    vector_index = (
        schema.get("keys", {}).get("#embedding", {})
        .get("float_list", {}).get("vector_index", {})
    )
    if vector_index.get("enabled") is not True:
        raise RuntimeError(f"No active embedding index: {collection_name}")
    actual = vector_index.get("config", {}).get("space")
    if actual != expected:
        raise RuntimeError(
            f"{collection_name}: persisted distance mismatch: "
            f"expected {expected!r}, got {actual!r}"
        )
    legacy = metadata.get("hnsw:space")
    if legacy is not None and legacy != actual:
        raise RuntimeError(f"Conflicting legacy distance metadata: {collection_name}")
    return actual


client = chromadb.PersistentClient(path=str(RUN_DB))
manifests, collections, manifest_paths = {}, {}, {}
for strategy in "ABC":
    paths = list((DATA_ROOT / "data/vector_store/manifests").glob(
        f"chunks_{strategy.lower()}_*.manifest.json"
    ))
    assert len(paths) == 1, f"{strategy}: expected exactly one manifest; select the version explicitly"
    manifest_paths[strategy] = paths[0]
    m = json.loads(paths[0].read_text(encoding="utf-8"))
    assert m["schema_version"] == "embedding_index.manifest.v1"
    assert m["input"]["strategy"] == strategy
    assert m["embedding"]["normalization"] == "L2"
    assert m["vector_store"]["distance"] == "cosine"
    chunk_path = DATA_ROOT / m["input"]["path"]
    assert sha256_file(chunk_path) == m["input"]["sha256"]
    with chunk_path.open(encoding="utf-8") as f:
        assert sum(bool(line.strip()) for line in f) == m["chunk_count"]
    collection = client.get_collection(m["vector_store"]["collection"], embedding_function=None)
    assert collection.count() == m["chunk_count"]
    metadata = collection.metadata or {}
    for key, expected in {
        "model_name":m["embedding"]["model"],
        "model_revision":m["embedding"]["revision"],
        "embedding_dimension":m["embedding"]["dimension"],
        "input_sha256":m["input"]["sha256"],
    }.items():
        assert metadata.get(key) == expected, f"{strategy}: metadata mismatch: {key}"
    # Chroma 1.5.9 persists distance in the vector schema, not user metadata.
    actual_distance = verify_persisted_distance(
        RUN_DB, collection.name, metadata, m["vector_store"]["distance"]
    )
    print(strategy, "Verified persisted distance:", actual_distance)
    saved = collection.get(limit=1, include=["embeddings", "documents", "metadatas"])
    vector = np.asarray(saved["embeddings"][0])
    assert vector.shape == (m["embedding"]["dimension"],)
    assert np.isfinite(vector).all() and np.isclose(np.linalg.norm(vector), 1, atol=1e-4)
    manifests[strategy], collections[strategy] = m, collection
    print(strategy, collection.name, collection.count(), "Checks passed")


## Initialization 3

In [ ]:
from sentence_transformers import SentenceTransformer
import re

embedding = manifests["B"]["embedding"]
assert embedding["model"] == "sentence-transformers/all-MiniLM-L6-v2"
assert embedding["revision"] == "1110a243fdf4706b3f48f1d95db1a4f5529b4d41"
assert re.fullmatch(r"[0-9a-f]{40}", embedding["revision"])
for m in manifests.values():
    assert m["embedding"] == embedding
started = time.perf_counter()
from huggingface_hub import snapshot_download

# Load the exact cached snapshot so its revision remains observable.
allow_download = os.environ.get("M4_ALLOW_MODEL_DOWNLOAD", "0") == "1"
snapshot = Path(snapshot_download(
    repo_id=embedding["model"], revision=embedding["revision"],
    cache_dir=str(BASE / "model_cache"), local_files_only=not allow_download,
    allow_patterns=["*.json", "*.safetensors", "vocab.txt"],
)).resolve()
assert snapshot.name == embedding["revision"]
model = SentenceTransformer(str(snapshot), device="cpu", local_files_only=True)
# Prefer the current API while supporting older sentence-transformers versions.
get_dimension = getattr(model, "get_embedding_dimension", None)
if not callable(get_dimension):
    get_dimension = model.get_sentence_embedding_dimension
assert get_dimension() == embedding["dimension"]
resolved = getattr(model._first_module().auto_model.config, "_commit_hash", None)
if resolved is not None:
    assert resolved == embedding["revision"], "Resolved model revision does not match"
probe = model.encode(["startup check"], normalize_embeddings=True, convert_to_numpy=True)
assert probe.shape == (1, embedding["dimension"])
assert np.isfinite(probe).all() and np.isclose(np.linalg.norm(probe[0]), 1, atol=1e-4)
print("Model ready; maximum input tokens:", model.max_seq_length)
print("Loading and warm-up time (seconds):", round(time.perf_counter() - started, 2))


## Initialization 4

In [ ]:
scripts_path = str(PROJECT_ROOT / "scripts")
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)
from m4_retrieval.retriever import Retriever
from m4_retrieval.workshop import Workshop, save_json
retriever = Retriever(model, collections, manifests, manifest_paths)
w = Workshop(retriever, PROJECT_ROOT, DATA_ROOT, BASE)
w.load_corpus()
print("HANDOFF READY")


## Integration verification
Run this cell before accepting the live runtime. Failures must be investigated, not suppressed.

In [ ]:
unit_result = w.unit_checks()
integration_result = w.integration_checks()
save_json(PROJECT_ROOT / 'documents/m4/delivery_live_checks.json', {'unit': unit_result, 'integration': integration_result})

## M5 retrieval example
Hybrid/C is an evaluation-informed demonstration choice, not independently validated after selection. Never treat OK as proof that the question is answerable.

In [ ]:
response = w.retrieve('Which celestial body prompted the 2006 vote on Pluto?', strategy='C', mode='hybrid', top_k=5, candidate_k=30, rrf_constant=60)
if response['status'] == 'ERROR':
    raise RuntimeError(response['error'])
selected, context, used = w.pack(response['results'], token_budget=1024)
save_json(PROJECT_ROOT / 'documents/m4/m5_demo.json', {'retrieval': response, 'context': context, 'context_tokens': used, 'selected_chunk_ids': [r['chunk_id'] for r in selected]})
print(response['status'], len(response['results']), used)
print(context)

## Optional: reproduce Dense comparison
This is an evaluation cell, not a tuning step. Run Dense first, then the supplementary comparison.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from statistics import mean, median
from importlib.metadata import version
import csv
import inspect
import json
import uuid
from m4_retrieval.workshop import save_json, digest, evidence_coverage

if "w" not in globals():
    raise RuntimeError("Restore sections 0A through 0D and 3A first.")

w.load_corpus()
w.validate_startup()
qa = w.load_qa()
questions = [
    q for q in qa
    if q["split"] == "test" and q["answerable"] is True
]
if len(questions) < 20:
    raise ValueError("At least 20 answerable test questions are required.")
if any(q.get("reviewed") is not True for q in questions):
    raise ValueError("All test questions must be reviewed.")
if any(not q.get("gold_evidence") for q in questions):
    raise ValueError("All answerable questions need gold evidence.")
if len({q["question_id"] for q in questions}) != len(questions):
    raise ValueError("Duplicate test IDs found.")

TOP_K = 5
TOKEN_BUDGET = 1024
STRATEGIES = ("A", "B", "C")

stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
eval_dir = Path(w.output) / f"dense_fixed_{stamp}_{uuid.uuid4().hex[:6]}"
eval_dir.mkdir(parents=True)

qa_hash = digest(w.qa_path)
metadata = {
    "status": "RUNNING",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "split": "test",
    "mode": "dense",
    "strategies": list(STRATEGIES),
    "top_k": TOP_K,
    "candidate_k": 30,
    "token_budget": TOKEN_BUDGET,
    "budget_tokenizer": "retriever embedding tokenizer; M5 proxy only",
    "question_count": len(questions),
    "qa_sha256": qa_hash,
    "schema_sha256": digest(w.schema_path),
    "index_hashes": {
        s: digest(w.manifest_paths[s]) for s in STRATEGIES
    },
    "embedding": {
        s: w.manifests[s]["embedding"] for s in STRATEGIES
    },
    "workshop_source_sha256": digest(inspect.getfile(type(w))),
    "packages": {
        name: version(name)
        for name in ("chromadb", "sentence-transformers", "numpy")
    },
    "protocol": "Fixed comparison; no test-driven parameter selection",
    "limitations": [
        "Answerable science questions only; no refusal evaluation",
        "Questions from the same article are correlated",
        "Recall measures annotated evidence units, not all relevant chunks",
    ],
}
save_json(eval_dir / "run_metadata.json", metadata)
save_json(eval_dir / "test_questions_snapshot.json", questions)

summary = []
try:
    with (eval_dir / "responses.jsonl").open("w", encoding="utf-8") as stream:
        for strategy in STRATEGIES:
            warmup = w.retrieve(
                questions[0]["question"], strategy=strategy,
                mode="dense", top_k=TOP_K, candidate_k=30,
            )
            if warmup["status"] == "ERROR":
                raise RuntimeError(warmup["error"])

            raw_scores, budget_scores, latencies, token_counts = [], [], [], []
            for number, question in enumerate(questions, start=1):
                response = w.retrieve(
                    question["question"], strategy=strategy,
                    mode="dense", top_k=TOP_K, candidate_k=30,
                )
                if response["status"] == "ERROR":
                    save_json(eval_dir / "failed_response.json", response)
                    raise RuntimeError(response["error"])

                selected, context, used = w.pack(
                    response["results"], token_budget=TOKEN_BUDGET
                )
                raw = evidence_coverage(question, response["results"], strategy)
                budget = evidence_coverage(question, selected, strategy)
                raw_scores.append(raw)
                budget_scores.append(budget)
                latencies.append(response["timing_ms"]["total"])
                token_counts.append(used)

                record = {
                    "question_id": question["question_id"],
                    "strategy": strategy,
                    "retrieval": response,
                    "raw_metrics": raw,
                    "budget_metrics": budget,
                    "context": context,
                    "context_tokens": used,
                    "selected_chunk_ids": [r["chunk_id"] for r in selected],
                }
                stream.write(json.dumps(record, ensure_ascii=False) + "\n")
                stream.flush()
                print(f"{strategy}: {number}/{len(questions)}", flush=True)

            row = {
                "mode": "dense", "strategy": strategy, "top_k": TOP_K,
                "question_count": len(questions), "error_count": 0,
                "raw_hit": mean(r["hit"] for r in raw_scores),
                "raw_recall": mean(r["recall"] for r in raw_scores),
                "raw_complete": mean(r["complete"] for r in raw_scores),
                "budget_hit": mean(r["hit"] for r in budget_scores),
                "budget_recall": mean(r["recall"] for r in budget_scores),
                "budget_complete": mean(r["complete"] for r in budget_scores),
                "latency_median_ms": median(latencies),
                "mean_context_tokens": mean(token_counts),
            }
            summary.append(row)
            save_json(eval_dir / "summary.partial.json", summary)
            print(
                f"{strategy} DONE | Hit@5={row['raw_hit']:.3f} | "
                f"Recall@5={row['raw_recall']:.3f} | "
                f"Budget recall={row['budget_recall']:.3f}",
                flush=True,
            )

    if digest(w.qa_path) != qa_hash:
        raise RuntimeError("QA file changed during evaluation.")
    save_json(eval_dir / "summary.json", summary)
    with (eval_dir / "summary.csv").open(
        "w", encoding="utf-8-sig", newline=""
    ) as stream:
        writer = csv.DictWriter(stream, fieldnames=list(summary[0]))
        writer.writeheader()
        writer.writerows(summary)
    metadata["status"] = "COMPLETE"
except Exception as error:
    metadata.update(status="FAILED", error=str(error))
    save_json(eval_dir / "run_metadata.json", metadata)
    raise

save_json(eval_dir / "run_metadata.json", metadata)
print("\nEVALUATION COMPLETE")
print("Results folder:", eval_dir)

## Optional: reproduce BM25 and hybrid comparison
This is an evaluation cell, not a tuning step. Run Dense first, then the supplementary comparison.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from statistics import mean, median
from IPython.display import display, Markdown
import csv
import json
import uuid
from m4_retrieval.workshop import save_json, digest, evidence_coverage

if "w" not in globals() or "eval_dir" not in globals():
    raise RuntimeError("Restore the workshop and the Dense results folder first.")

baseline_dir = Path(eval_dir)
metadata = json.loads(
    (baseline_dir / "run_metadata.json").read_text(encoding="utf-8")
)
baseline = json.loads(
    (baseline_dir / "summary.json").read_text(encoding="utf-8")
)
if metadata.get("status") != "COMPLETE":
    raise ValueError("The Dense baseline is incomplete.")
if metadata["qa_sha256"] != digest(w.qa_path):
    raise ValueError("QA changed after the Dense baseline. Stop for review.")
if metadata["schema_sha256"] != digest(w.schema_path):
    raise ValueError("Schema changed after the Dense baseline.")
for strategy in metadata["strategies"]:
    if metadata["index_hashes"][strategy] != digest(w.manifest_paths[strategy]):
        raise ValueError("Index changed: " + strategy)

questions = json.loads(
    (baseline_dir / "test_questions_snapshot.json").read_text(encoding="utf-8")
)
TOP_K = metadata["top_k"]
TOKEN_BUDGET = metadata["token_budget"]
STRATEGIES = metadata["strategies"]

comparison_dir = baseline_dir / ("lexical_comparison_" + uuid.uuid4().hex[:8])
comparison_dir.mkdir()
run_metadata = {
    **metadata,
    "status": "RUNNING",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "mode": "bm25_and_hybrid",
    "baseline_folder": str(baseline_dir),
    "candidate_k": 30,
    "rrf_constant": 60,
    "bm25": "positive log1p IDF; k1=1.5; b=0.75",
    "protocol": "Supplementary fixed comparison; no parameter tuning",
}
save_json(comparison_dir / "run_metadata.json", run_metadata)
comparison_summary = list(baseline)

try:
    print("Preparing BM25 indexes. Please wait.", flush=True)
    w.build_bm25()
    print("BM25 indexes ready.", flush=True)

    with (comparison_dir / "responses.jsonl").open("w", encoding="utf-8") as stream:
        for mode in ("bm25", "hybrid"):
            for strategy in STRATEGIES:
                warmup = w.retrieve(
                    questions[0]["question"], mode=mode, strategy=strategy,
                    top_k=TOP_K, candidate_k=30, rrf_constant=60,
                )
                if warmup["status"] == "ERROR":
                    raise RuntimeError(warmup["error"])

                raw_scores, budget_scores, latencies, tokens = [], [], [], []
                for number, question in enumerate(questions, start=1):
                    response = w.retrieve(
                        question["question"], mode=mode, strategy=strategy,
                        top_k=TOP_K, candidate_k=30, rrf_constant=60,
                    )
                    if response["status"] == "ERROR":
                        save_json(comparison_dir / "failed_response.json", response)
                        raise RuntimeError(response["error"])

                    selected, context, used = w.pack(
                        response["results"], token_budget=TOKEN_BUDGET
                    )
                    raw = evidence_coverage(question, response["results"], strategy)
                    budget = evidence_coverage(question, selected, strategy)
                    raw_scores.append(raw)
                    budget_scores.append(budget)
                    latencies.append(response["timing_ms"]["total"])
                    tokens.append(used)

                    stream.write(json.dumps({
                        "question_id": question["question_id"],
                        "mode": mode,
                        "strategy": strategy,
                        "retrieval": response,
                        "raw_metrics": raw,
                        "budget_metrics": budget,
                        "context": context,
                        "context_tokens": used,
                        "selected_chunk_ids": [r["chunk_id"] for r in selected],
                    }, ensure_ascii=False) + "\n")
                    stream.flush()
                    print(
                        f"{mode} {strategy}: {number}/{len(questions)}",
                        flush=True,
                    )

                row = {
                    "mode": mode,
                    "strategy": strategy,
                    "top_k": TOP_K,
                    "question_count": len(questions),
                    "error_count": 0,
                    "raw_hit": mean(r["hit"] for r in raw_scores),
                    "raw_recall": mean(r["recall"] for r in raw_scores),
                    "raw_complete": mean(r["complete"] for r in raw_scores),
                    "budget_hit": mean(r["hit"] for r in budget_scores),
                    "budget_recall": mean(r["recall"] for r in budget_scores),
                    "budget_complete": mean(r["complete"] for r in budget_scores),
                    "latency_median_ms": median(latencies),
                    "mean_context_tokens": mean(tokens),
                }
                comparison_summary.append(row)
                save_json(comparison_dir / "summary.partial.json", comparison_summary)

    if digest(w.qa_path) != metadata["qa_sha256"]:
        raise RuntimeError("QA changed during evaluation.")
    save_json(comparison_dir / "summary.json", comparison_summary)
    with (comparison_dir / "summary.csv").open(
        "w", encoding="utf-8-sig", newline=""
    ) as stream:
        writer = csv.DictWriter(stream, fieldnames=list(comparison_summary[0]))
        writer.writeheader()
        writer.writerows(comparison_summary)
    run_metadata["status"] = "COMPLETE"
except Exception as error:
    run_metadata.update(status="FAILED", error=str(error))
    save_json(comparison_dir / "run_metadata.json", run_metadata)
    raise

save_json(comparison_dir / "run_metadata.json", run_metadata)
lines = [
    "| Mode | Strategy | Hit@5 | Evidence Recall@5 | Budget Recall |",
    "|---|---|---:|---:|---:|",
]
for row in comparison_summary:
    lines.append(
        f"| {row['mode']} | {row['strategy']} | {row['raw_hit']:.1%} | "
        f"{row['raw_recall']:.1%} | {row['budget_recall']:.1%} |"
    )
table = "\n".join(lines)
(comparison_dir / "results_table.md").write_text(table, encoding="utf-8")
display(Markdown(table))
print("COMPARISON COMPLETE")
print("Results folder:", comparison_dir)